# E11 — YOLO Hybrid

**Owner:** Nada

**Concept:** Augment YOLOv8 FPN features with classical 252-dim features to improve classification head.

**Dependencies:** E1 weights (`models/yolo/yolo_baseline_E1.pt`) + `features/classical/`

In [1]:
import sys, os, time, json, datetime, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, balanced_accuracy_score, roc_auc_score,
    matthews_corrcoef, cohen_kappa_score, top_k_accuracy_score,
)

# ── Locked constants ──
SEED = 42
CLASSES = ["BIODEGRADABLE", "CARDBOARD", "GLASS", "METAL", "PAPER", "PLASTIC"]
N_CLASSES = 6
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")


Device: cpu
PyTorch: 2.12.0


In [2]:
# Adjust REPO_ROOT if needed
REPO_ROOT = os.path.dirname(os.path.abspath("__file__"))
if not os.path.isdir(os.path.join(REPO_ROOT, "src")):
    REPO_ROOT = os.path.dirname(REPO_ROOT)

FEAT_DIR = os.path.join(REPO_ROOT, "data", "processed", "features")
MDL_DIR  = os.path.join(REPO_ROOT, "models");        os.makedirs(MDL_DIR, exist_ok=True)
RES_DIR  = os.path.join(REPO_ROOT, "results", "metrics"); os.makedirs(RES_DIR, exist_ok=True)
FIG_DIR  = os.path.join(REPO_ROOT, "figures", "fusion");  os.makedirs(FIG_DIR, exist_ok=True)
PRED_DIR = os.path.join(REPO_ROOT, "results", "fusion");  os.makedirs(PRED_DIR, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Features:  {FEAT_DIR}")


Repo root: /Users/nadaashraf/Desktop/CV-Project
Features:  /Users/nadaashraf/Desktop/CV-Project/data/processed/features


In [3]:
print("Loading features...")
t0 = time.time()

# YOLO SPPF features (256-dim) — from Aly's E1
X_y_train = np.load(os.path.join(FEAT_DIR, "yolo_train_X.npy"))
X_y_val   = np.load(os.path.join(FEAT_DIR, "yolo_val_X.npy"))
X_y_test  = np.load(os.path.join(FEAT_DIR, "yolo_test_X.npy"))

# Classical features (252-dim) — from Aly's E2  (use CLEAN, not augmented!)
X_c_train = np.load(os.path.join(FEAT_DIR, "classical_train_clean_X.npy"))
X_c_val   = np.load(os.path.join(FEAT_DIR, "classical_val_X.npy"))
X_c_test  = np.load(os.path.join(FEAT_DIR, "classical_test_X.npy"))

# Labels
y_train = np.load(os.path.join(FEAT_DIR, "classical_train_clean_y.npy"))
y_val   = np.load(os.path.join(FEAT_DIR, "classical_val_y.npy"))
y_test  = np.load(os.path.join(FEAT_DIR, "classical_test_y.npy"))

print(f"Loaded in {time.time()-t0:.1f}s")
print(f"  YOLO   train: {X_y_train.shape}  val: {X_y_val.shape}  test: {X_y_test.shape}")
print(f"  Classic train: {X_c_train.shape}  val: {X_c_val.shape}  test: {X_c_test.shape}")
print(f"  Labels train: {y_train.shape}  val: {y_val.shape}  test: {y_test.shape}")


Loading features...
Loaded in 0.1s
  YOLO   train: (45177, 256)  val: (9935, 256)  test: (10553, 256)
  Classic train: (45177, 252)  val: (9935, 252)  test: (10553, 252)
  Labels train: (45177,)  val: (9935,)  test: (10553,)


In [4]:
# Row alignment — shapes must match
assert X_y_train.shape[0] == X_c_train.shape[0] == y_train.shape[0], "Train row mismatch!"
assert X_y_val.shape[0]   == X_c_val.shape[0]   == y_val.shape[0],   "Val row mismatch!"
assert X_y_test.shape[0]  == X_c_test.shape[0]   == y_test.shape[0],  "Test row mismatch!"
assert X_y_train.shape[1] == 256, f"YOLO dim should be 256, got {X_y_train.shape[1]}"
assert X_c_train.shape[1] == 252, f"Classical dim should be 252, got {X_c_train.shape[1]}"

print("✅ All checks passed")
print(f"\n  Train label distribution:")
for i, cls in enumerate(CLASSES):
    n = np.sum(y_train == i)
    print(f"    {i} {cls:15s}: {n:6d} ({100*n/len(y_train):.1f}%)")


✅ All checks passed

  Train label distribution:
    0 BIODEGRADABLE  :  26688 (59.1%)
    1 CARDBOARD      :   3225 (7.1%)
    2 GLASS          :   4666 (10.3%)
    3 METAL          :   3643 (8.1%)
    4 PAPER          :   2828 (6.3%)
    5 PLASTIC        :   4127 (9.1%)


In [5]:
# StandardScaler — fit on train ONLY
sc_yolo = StandardScaler().fit(X_y_train)
sc_cls  = StandardScaler().fit(X_c_train)

Xy_tr = sc_yolo.transform(X_y_train).astype(np.float32)
Xy_va = sc_yolo.transform(X_y_val).astype(np.float32)
Xy_te = sc_yolo.transform(X_y_test).astype(np.float32)

Xc_tr = sc_cls.transform(X_c_train).astype(np.float32)
Xc_va = sc_cls.transform(X_c_val).astype(np.float32)
Xc_te = sc_cls.transform(X_c_test).astype(np.float32)

# Fuse: YOLO (256) + Classical (252) = 508-dim
X_fused_tr = np.hstack([Xy_tr, Xc_tr])  # (45177, 508)
X_fused_va = np.hstack([Xy_va, Xc_va])
X_fused_te = np.hstack([Xy_te, Xc_te])

FUSED_DIM = X_fused_tr.shape[1]
print(f"✅ Fused shape: {X_fused_tr.shape}  ({FUSED_DIM} = 256 YOLO + 252 Classical)")

# Save scalers
joblib.dump(sc_yolo, os.path.join(MDL_DIR, "e11_scaler_yolo.pkl"))
joblib.dump(sc_cls,  os.path.join(MDL_DIR, "e11_scaler_classical.pkl"))
print("  Scalers saved")


✅ Fused shape: (45177, 508)  (508 = 256 YOLO + 252 Classical)
  Scalers saved


In [6]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("Training SVM on Fused Features (508-dim)...")
print("Note: SVM training on 45,000 samples might take 2-5 minutes.")
t0 = time.time()

# We use the exact same SVM parameters that won E3 and E5
model = SVC(
    kernel="rbf",
    C=10.0,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=SEED
)

# Fit on training data
model.fit(X_fused_tr, y_train)
train_time = time.time() - t0
print(f"Training finished in {train_time:.1f} seconds.")

# --- Evaluate on Validation Set ---
print("\nEvaluating on Validation Set...")
val_preds = model.predict(X_fused_va)

val_acc = accuracy_score(y_val, val_preds)
val_f1 = f1_score(y_val, val_preds, average="macro", zero_division=0)

# We save best_f1 so the rest of the notebook cells don't break
best_f1 = val_f1  

print(f"✅ Val Accuracy: {val_acc:.4f}")
print(f"✅ Val Macro-F1: {val_f1:.4f}")


Training SVM on Fused Features (508-dim)...
Note: SVM training on 45,000 samples might take 2-5 minutes.
Training finished in 1611.3 seconds.

Evaluating on Validation Set...
✅ Val Accuracy: 0.8355
✅ Val Macro-F1: 0.7236


In [7]:
print("Evaluating on Test Set...")
t0 = time.time()
preds_te = model.predict(X_fused_te)
proba_te = model.predict_proba(X_fused_te)
clf_time = time.time() - t0

test_acc = accuracy_score(y_test, preds_te)
test_f1  = f1_score(y_test, preds_te, average="macro", zero_division=0)
test_wf1 = f1_score(y_test, preds_te, average="weighted", zero_division=0)
test_bal = balanced_accuracy_score(y_test, preds_te)
test_mcc = matthews_corrcoef(y_test, preds_te)
test_kap = cohen_kappa_score(y_test, preds_te)

try:
    test_auc = roc_auc_score(y_test, proba_te, multi_class="ovr", average="macro")
except: test_auc = float("nan")

try:
    test_top2 = top_k_accuracy_score(y_test, proba_te, k=2)
except: test_top2 = float("nan")

print("=" * 60)
print("E11 — YOLO Hybrid Results (SVM Version - Test Set)")
print("=" * 60)
print(f"  Test Accuracy  : {test_acc:.4f}  ← Overall Accuracy")
print(f"  Test Macro-F1  : {test_f1:.4f}  ← Primary Metric")
print(f"  Weighted-F1    : {test_wf1:.4f}")
print(f"  Balanced Acc   : {test_bal:.4f}")
print(f"  MCC            : {test_mcc:.4f}")
print(f"  Cohen Kappa    : {test_kap:.4f}")
print(f"  AUC-ROC (macro): {test_auc:.4f}")
print(f"  Feature dim    : {FUSED_DIM}")
print(f"  Inference      : {clf_time/len(y_test)*1000:.4f} ms/crop")

per_f1   = f1_score(y_test, preds_te, average=None, labels=list(range(N_CLASSES)), zero_division=0)
per_prec = precision_score(y_test, preds_te, average=None, labels=list(range(N_CLASSES)), zero_division=0)
per_rec  = recall_score(y_test, preds_te, average=None, labels=list(range(N_CLASSES)), zero_division=0)

print(f"\n  Per-class F1:")
for i, cls in enumerate(CLASSES):
    print(f"    {cls:15s}: F1={per_f1[i]:.4f}  Prec={per_prec[i]:.4f}  Rec={per_rec[i]:.4f}")


Evaluating on Test Set...
E11 — YOLO Hybrid Results (SVM Version - Test Set)
  Test Accuracy  : 0.8253  ← Overall Accuracy
  Test Macro-F1  : 0.7345  ← Primary Metric
  Weighted-F1    : 0.8166
  Balanced Acc   : 0.7107
  MCC            : 0.7225
  Cohen Kappa    : 0.7170
  AUC-ROC (macro): 0.9452
  Feature dim    : 508
  Inference      : 5.6177 ms/crop

  Per-class F1:
    BIODEGRADABLE  : F1=0.9181  Prec=0.8670  Rec=0.9756
    CARDBOARD      : F1=0.8455  Prec=0.8418  Rec=0.8492
    GLASS          : F1=0.6677  Prec=0.8321  Rec=0.5576
    METAL          : F1=0.6842  Prec=0.7491  Rec=0.6297
    PAPER          : F1=0.6523  Prec=0.7158  Rec=0.5991
    PLASTIC        : F1=0.6393  Prec=0.6263  Rec=0.6529


In [8]:
print("\n" + "=" * 60)
print("COMPARISON vs Baselines")
print("=" * 60)
baselines = {
    "E2  Classical RF (252)":  0.6476,
    "E3  Deep SVM (1280)":    0.7848,
    "E4  Raw Fusion RF (1532)": 0.6960,
    "E5  PCA Fusion SVM (939)": 0.8071,
    "E7  Feat Sel RF (200)":  0.7079,
    "E8  Avg Voting":         0.7820,
    "E9  Weighted Voting":    0.7832,
    "E10 Attention Fusion":   0.7752,
}
for name, mf1 in baselines.items():
    delta = test_f1 - mf1
    flag = "✅" if delta > 0 else "  "
    print(f"  {flag} {name}: {mf1:.4f}  (E11 delta: {delta:+.4f})")
print(f"\n  E11 YOLO Hybrid ({FUSED_DIM}): {test_f1:.4f}")



COMPARISON vs Baselines
  ✅ E2  Classical RF (252): 0.6476  (E11 delta: +0.0869)
     E3  Deep SVM (1280): 0.7848  (E11 delta: -0.0503)
  ✅ E4  Raw Fusion RF (1532): 0.6960  (E11 delta: +0.0385)
     E5  PCA Fusion SVM (939): 0.8071  (E11 delta: -0.0726)
  ✅ E7  Feat Sel RF (200): 0.7079  (E11 delta: +0.0266)
     E8  Avg Voting: 0.7820  (E11 delta: -0.0475)
     E9  Weighted Voting: 0.7832  (E11 delta: -0.0487)
     E10 Attention Fusion: 0.7752  (E11 delta: -0.0407)

  E11 YOLO Hybrid (508): 0.7345


In [9]:
cm = confusion_matrix(y_test, preds_te, labels=list(range(N_CLASSES)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
axes[0].set_title("E11 — Confusion Matrix (counts)")

# Normalized
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
axes[1].set_title("E11 — Confusion Matrix (normalized)")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "E11_confusion_matrix.png"), dpi=150)
plt.show()
print("✅ Saved")


✅ Saved


In [10]:
e3_f1s = [0.9249, 0.8706, 0.7294, 0.7654, 0.7419, 0.6767]  # from Aly's report

x = np.arange(N_CLASSES)
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, e3_f1s, width, label="E3 Deep SVM", color="#4C72B0", edgecolor="white")
bars2 = ax.bar(x + width/2, per_f1, width, label="E11 YOLO Hybrid", color="#DD8452", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=15)
ax.set_ylabel("F1 Score")
ax.set_title("E11 vs E3 — Per-Class F1")
ax.legend()
ax.set_ylim(0.4, 1.0)

for bar, val in zip(bars1, e3_f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.3f}", ha="center", fontsize=7)
for bar, val in zip(bars2, per_f1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.3f}", ha="center", fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "E11_vs_E3_perclass_f1.png"), dpi=150)
plt.show()


In [11]:
# ── Save model ──
model_path = os.path.join(MDL_DIR, "e11_hybrid_svm_model.pkl")
joblib.dump(model, model_path)
model_mb = os.path.getsize(model_path) / 1e6
print(f"Model saved: {model_path} ({model_mb:.2f} MB)")

# ── Save predictions ──
np.save(os.path.join(PRED_DIR, "e11_predictions.npy"), preds_te)
np.save(os.path.join(PRED_DIR, "e11_probabilities.npy"), proba_te)

# ── Save per-class metrics CSV ──
pc_rows = []
for i, cls in enumerate(CLASSES):
    pc_rows.append({"class": cls, "f1": round(float(per_f1[i]), 4),
                     "precision": round(float(per_prec[i]), 4),
                     "recall": round(float(per_rec[i]), 4)})
pd.DataFrame(pc_rows).to_csv(os.path.join(RES_DIR, "E11_per_class_metrics.csv"), index=False)

# ── Append to shared classification_results.csv ──
row = {
    "experiment": "E11", "model": "HybridSVM_YOLO+Classical",
    "feature_dim": FUSED_DIM,
    "accuracy": round(test_acc, 4), "macro_f1": round(test_f1, 4),
    "weighted_f1": round(test_wf1, 4), "balanced_accuracy": round(test_bal, 4),
    "mcc": round(test_mcc, 4), "cohen_kappa": round(test_kap, 4),
    "auc_roc_macro": round(test_auc, 4) if not np.isnan(test_auc) else float("nan"),
    "val_macro_f1": round(best_f1, 4),
    "model_size_mb": round(model_mb, 4),
    "inference_ms_per_crop": round(clf_time/len(y_test)*1000, 4),
    **{f"f1_{CLASSES[i]}": round(float(per_f1[i]), 4) for i in range(N_CLASSES)},
    **{f"prec_{CLASSES[i]}": round(float(per_prec[i]), 4) for i in range(N_CLASSES)},
    **{f"rec_{CLASSES[i]}": round(float(per_rec[i]), 4) for i in range(N_CLASSES)},
    "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
}

csv_path = os.path.join(RES_DIR, "classification_results.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df = df[df["experiment"] != "E11"]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
else:
    df = pd.DataFrame([row])
df.to_csv(csv_path, index=False)

print("\n✅ All outputs saved:")
print(f"  Model         → {model_path}")
print(f"  Predictions   → {PRED_DIR}/e11_predictions.npy")
print(f"  Probabilities → {PRED_DIR}/e11_probabilities.npy")
print(f"  Per-class     → {RES_DIR}/E11_per_class_metrics.csv")
print(f"  Results CSV   → {csv_path}")
print(f"\n🎉 E11 YOLO Hybrid (SVM) complete!")


Model saved: /Users/nadaashraf/Desktop/CV-Project/models/e11_hybrid_svm_model.pkl (83.15 MB)

✅ All outputs saved:
  Model         → /Users/nadaashraf/Desktop/CV-Project/models/e11_hybrid_svm_model.pkl
  Predictions   → /Users/nadaashraf/Desktop/CV-Project/results/fusion/e11_predictions.npy
  Probabilities → /Users/nadaashraf/Desktop/CV-Project/results/fusion/e11_probabilities.npy
  Per-class     → /Users/nadaashraf/Desktop/CV-Project/results/metrics/E11_per_class_metrics.csv
  Results CSV   → /Users/nadaashraf/Desktop/CV-Project/results/metrics/classification_results.csv

🎉 E11 YOLO Hybrid (SVM) complete!
